# Neural networks in practice: object detection with YOLO

So far our machine learning has been *unsupervised* (clustering, PCA). Now we turn
to a **supervised** neural network trained on labelled data.

[YOLO](https://docs.ultralytics.com/) ("You Only Look Once") is a family of
convolutional neural networks for **object detection**: given an image, it draws a
box around each object it finds and labels it. The version we use, `yolo11n`, is
the "nano" model — small enough to run on a laptop CPU — and was trained on the
[COCO](https://cocodataset.org/) dataset of 80 everyday object categories.

Related tools built on the same ideas are used throughout biology, for example
[DeepLabCut](https://www.mackenziemathislab.org/deeplabcut) for tracking animal
body parts in behavioural experiments.

## Load a pre-trained model

The first time you run this, the model weights (about 5 MB) are downloaded
automatically and cached, so this cell needs an internet connection the first
time. We do not train the network ourselves — training YOLO from scratch needs
many labelled images and a lot of compute. Instead we do **inference** with a
network someone else already trained.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")   # downloads the trained weights on first use

## Run detection on an image

Ultralytics ships a couple of example images. We run the detector on one of them.
Setting `verbose=False` just keeps the output tidy.

In [ ]:
from ultralytics.utils import ASSETS

results = model(ASSETS / "bus.jpg", verbose=False)
result = results[0]

Each detection has a class label and a confidence. Let's count what was found:

In [ ]:
from collections import Counter

labels = [result.names[int(c)] for c in result.boxes.cls]
Counter(labels)

## Look at the result

`result.plot()` returns the image with the predicted boxes and labels drawn on it.
It comes back in BGR channel order (an OpenCV convention), so we flip it to RGB
before showing it with matplotlib.

In [ ]:
import matplotlib.pyplot as plt

annotated_bgr = result.plot()
annotated_rgb = annotated_bgr[:, :, ::-1]   # BGR -> RGB

plt.figure(figsize=(6, 8))
plt.imshow(annotated_rgb)
plt.axis("off")
plt.show()

## Inspecting the predictions

The boxes carry the coordinates and confidences too. Here are the first few, with
their label and how confident the model was:

In [ ]:
for box in result.boxes[:5]:
    label = result.names[int(box.cls)]
    confidence = float(box.conf)
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    print(f"{label:10s}  confidence={confidence:.2f}  box=({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f})")

## Connecting the pieces

YOLO, the embeddings from the previous notebook, and the LLMs behind the AI
assistants from Lecture 4 are all **neural networks**: large functions with many
numeric parameters, tuned on data to map inputs to useful outputs. What changes
between them is the kind of data and the network architecture — pixels and
convolutions here, text and transformers there — but the underlying recipe of
"learn parameters from labelled or unlabelled data, then run inference" is shared.

Try running the detector on `ASSETS / "zidane.jpg"`, or on one of your own images,
and see what it finds.